# Global Pollution Analysis and Energy Recovery
## Applied Machine Learning: Exploratory Data Analysis & Predictive Modeling

### Project Overview
This project analyzes global environmental pollution (Air, Water, and Soil indices), industrial emissions, energy metrics, and macroeconomic indicators across 175 countries from 2000 to 2019. The dual modeling objectives are:
1. **Regression:** Predict waste-to-energy recovery in Gigawatt-hours (`Energy_Recovered (in GWh)`).
2. **Classification:** Predict ambient air pollution severity categorized into Low, Medium, and High classes without target leakage.

In this notebook, we follow strict data science discipline: ensuring data integrity, documenting anomalies, avoiding data leakage, benchmarking against baselines, and grounding all findings in empirical evidence.

---
### Notebook Outline
1. **Environment Setup & Configuration**
2. **Data Loading and Initial Inspection**
3. **Data Cleaning, Integrity & Anomaly Audit (Spain 2004 Investigation)**
4. **Descriptive Statistical Analysis**
5. **Exploratory Data Analysis (EDA) & Interpretations**
   - 5.1 Correlation Heatmap
   - 5.2 Pollution Distribution & Central Tendency (Boxplots)
   - 5.3 Pollution Trajectories Over Time (2000–2019)
   - 5.4 Cross-Country Pollution Comparisons
   - 5.5 Industrial & Plastic Waste Distributions
   - 5.6 Energy Recovery Target Distribution & Bivariate Relationships
6. **Feature Engineering & Mathematical Derivations**
   - 6.1 Total Energy Consumption Calculation & Unit Verification
   - 6.2 Waste-Per-Capita Indicators & Denominator Audit
   - 6.3 Feature Sets Formulation (Base vs. Engineered)
7. **Train/Test Splitting Strategies & Leakage Prevention**
   - 7.1 Evaluation Strategy 1: Random 80/20 Split
   - 7.2 Evaluation Strategy 2: Chronological Split (2000–2014 vs. 2015–2019)
   - 7.3 Target Formulation & Train-Only Tertile Thresholding
   - 7.4 Comprehensive Anti-Leakage Verification Checks
8. **Predictive Modeling: Linear Regression for Energy Recovery**
   - 8.1 Base vs. Engineered Models & Dummy Baseline
   - 8.2 Random vs. Chronological Evaluation (MAE, MSE, RMSE, R²)
   - 8.3 Regression Diagnostic Visualizations (Actual vs. Predicted, Residuals)
   - 8.4 Regression Performance Interpretation & Discussion
9. **Predictive Modeling: Logistic Regression for Pollution Severity**
   - 9.1 Multiclass Classification Pipeline & Dummy Baseline
   - 9.2 Random vs. Chronological Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC)
   - 9.3 Confusion Matrix Analysis & Visualizations
   - 9.4 Classification Performance Interpretation & Anti-Leakage Discussion
10. **Synthesis, Methodological Discipline, and Summary**

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Configure environment and styling
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Directory for persisted visual artifacts
VIZ_DIR = '../outputs/visualizations'
os.makedirs(VIZ_DIR, exist_ok=True)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Visualizations directory: {os.path.abspath(VIZ_DIR)}")

Pandas version: 3.1.1
NumPy version: 2.5.4
Visualizations directory: /Users/paarthrana/ML-from-scratch/Global-Pollution-Analysis/outputs/visualizations


## 1. Data Loading and Initial Inspection
We load the raw dataset from `../data/raw/Global_Pollution_Analysis.csv`. We verify the dimensions, column types, and sample records directly from the file without assumptions.

In [2]:
DATA_PATH = '../data/raw/Global_Pollution_Analysis.csv'
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("Column Information and Data Types:")
df.info()

Dataset Dimensions: 200 rows, 13 columns

Column Information and Data Types:
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Country                                 200 non-null    str    
 1   Year                                    200 non-null    int64  
 2   Air_Pollution_Index                     200 non-null    float64
 3   Water_Pollution_Index                   200 non-null    float64
 4   Soil_Pollution_Index                    200 non-null    float64
 5   Industrial_Waste (in tons)              200 non-null    float64
 6   Energy_Recovered (in GWh)               200 non-null    float64
 7   CO2_Emissions (in MT)                   200 non-null    float64
 8   Renewable_Energy (%)                    200 non-null    float64
 9   Plastic_Waste_Produced (in tons)        200 non-null    float64
 

In [3]:
df.head(5)

,Country,Year,Air_Pollution_Index,Water_Pollution_Index,Soil_Pollution_Index,Industrial_Waste (in tons),Energy_Recovered (in GWh),CO2_Emissions (in MT),Renewable_Energy (%),Plastic_Waste_Produced (in tons),Energy_Consumption_Per_Capita (in MWh),Population (in millions),GDP_Per_Capita (in USD)
0,Hungary,2005,272.70,124.27,51.95,94802.83,158.14,5.30,41.11,37078.88,12.56,42.22,20972.96
1,Singapore,2001,86.72,60.34,117.22,56283.92,498.04,6.34,36.44,33128.20,5.23,137.25,34850.41
2,Romania,2016,91.59,83.36,121.72,56256.02,489.51,49.69,9.38,18803.46,13.15,124.47,57773.15
3,Cook Islands,2018,280.61,67.16,93.58,74864.73,145.18,8.91,18.97,9182.27,0.78,67.80,21837.51
4,Djibouti,2008,179.16,127.53,121.55,76862.06,40.38,14.93,34.00,39235.12,12.84,186.52,41379.37


## 2. Data Cleaning and Integrity Audit
Before computing summary statistics, we verify data integrity across three criteria:
1. **Missing / Null values:** Any unrecorded or corrupted entries.
2. **Full-row duplicate records:** Identical duplicate entries across all 13 attributes.
3. **Compound key uniqueness:** Checking whether `(Country, Year)` uniquely identifies each observation.

In [4]:
# 1. Missing Value Audit
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print("=== Missing Value Audit ===")
print(missing_summary)

# 2. Exact Duplicate Rows
exact_dupes = df.duplicated().sum()
print(f"\nExact duplicate rows across all columns: {exact_dupes}")

# 3. Compound Key Duplicate Check (Country, Year)
cy_dupes = df.duplicated(subset=['Country', 'Year'], keep=False).sum()
print(f"Records sharing duplicate (Country, Year) keys: {cy_dupes}")

=== Missing Value Audit ===
                                        Missing_Count  Missing_Percentage (%)
Country                                             0                     0.0
Year                                                0                     0.0
Air_Pollution_Index                                 0                     0.0
Water_Pollution_Index                               0                     0.0
Soil_Pollution_Index                                0                     0.0
Industrial_Waste (in tons)                          0                     0.0
Energy_Recovered (in GWh)                           0                     0.0
CO2_Emissions (in MT)                               0                     0.0
Renewable_Energy (%)                                0                     0.0
Plastic_Waste_Produced (in tons)                    0                     0.0
Energy_Consumption_Per_Capita (in MWh)              0                     0.0
Population (in millions)            

### Spain 2004 Anomaly: Column-by-Column Investigation
While full-row duplicates are 0, exactly one compound key `(Spain, 2004)` appears twice in the dataset. Below, we conduct a column-by-column audit of both records.

In [5]:
spain_records = df[(df['Country'] == 'Spain') & (df['Year'] == 2004)]
audit_comparison = spain_records.T.rename(
    columns={spain_records.index[0]: 'Record A (Index 147)', spain_records.index[1]: 'Record B (Index 179)'}
)
audit_comparison['Difference / Note'] = audit_comparison.apply(
    lambda row: 'Identical' if row['Record A (Index 147)'] == row['Record B (Index 179)'] else 'Drastically Different', axis=1
)
audit_comparison

,Record A (Index 147),Record B (Index 179),Difference / Note
Country,Spain,Spain,Identical
Year,2004,2004,Identical
Air_Pollution_Index,217.42,134.92,Drastically Different
Water_Pollution_Index,97.5,157.31,Drastically Different
Soil_Pollution_Index,18.87,75.53,Drastically Different
Industrial_Waste (in tons),29480.82,88456.83,Drastically Different
Energy_Recovered (in GWh),54.68,307.91,Drastically Different
CO2_Emissions (in MT),43.72,2.42,Drastically Different
Renewable_Energy (%),31.11,18.68,Drastically Different
Plastic_Waste_Produced (in tons),49584.27,48265.22,Drastically Different


**Spain 2004 Investigation Findings & Decision:**
1. **Discrepancy across all numerical dimensions:** Every single numerical metric differs between Record A and Record B. For example:
   - Population differs by 40.23 million (105.76M vs. 145.99M).
   - GDP per capita differs by almost 2x ($49,171.17 vs. $26,353.46).
   - Industrial Waste differs by 3x (29,480.82 tons vs. 88,456.83 tons).
   - Energy Recovered differs by more than 5x (54.68 GWh vs. 307.91 GWh).
2. **Context & Demographic Plausibility:** Neither population figure corresponds to historical 2004 Spanish demographics (Spain's real population in 2004 was ~43.5 million; its nominal GDP per capita was ~$25,000 USD).
3. **Methodological Handling:** Without documented provenance indicating separate regional reporting or distinct survey methods, we **do not fabricate assumptions** or speculate. We do not automatically average or delete either observation. Both records are **retained initially** to preserve raw sample size ($N=200$).

### Topology of the Dataset: Country Cardinality and Temporal Structure
We inspect the frequency of observations across countries to determine whether this dataset represents a longitudinal panel or pooled cross-sectional data.

In [6]:
n_unique_countries = df['Country'].nunique()
counts_per_country = df['Country'].value_counts()

print(f"Total Unique Countries: {n_unique_countries} out of {len(df)} total observations.")
print("\nBreakdown of observations per country:")
counts_per_country.value_counts().rename_axis("Observations per Country").reset_index(name="Number of Countries")

Total Unique Countries: 175 out of 200 total observations.

Breakdown of observations per country:


,Observations per Country,Number of Countries
0,1,152
1,2,21
2,3,2


**Methodological Decision on the `Country` Feature:**
- **152 out of 175 countries appear exactly once.** Only 21 countries appear twice, and 2 countries (Moldova and Latvia) appear three times.
- **High-Cardinality Limitation:** One-hot encoding `Country` would generate 174 sparse dummy columns for just 200 total samples (a feature-to-sample ratio exceeding $0.87$). This would cause catastrophic overfitting and multicollinearity.
- **Generalization:** A model relying on country dummies cannot generalize to unseen nations or out-of-sample data.
- **Decision:** Following senior data science discipline, `Country` is treated as an observational entity identifier and **excluded from the primary predictor set** in both regression and classification models.

## 3. Descriptive Statistical Analysis
We calculate the standard measures of central tendency, dispersion, quantiles, and skewness for all numerical features.

In [7]:
num_cols = df.select_dtypes(include=[np.number]).columns
desc_stats = df[num_cols].describe(percentiles=[0.25, 0.50, 0.75]).T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc_stats['IQR'] = desc_stats['75%'] - desc_stats['25%']
desc_stats['Skewness'] = df[num_cols].skew()
desc_stats['Kurtosis'] = df[num_cols].kurtosis()
desc_stats.round(2)

,count,mean,std,min,25%,50%,75%,max,IQR,Skewness,Kurtosis
Year,200.0,2009.34,5.77,2000.00,2004.00,2010.00,2014.00,2019.00,10.00,-0.00,-1.26
Air_Pollution_Index,200.0,180.63,67.07,50.30,134.97,183.38,237.42,297.95,102.45,-0.13,-0.95
Water_Pollution_Index,200.0,115.07,47.58,31.13,74.55,112.30,157.48,199.32,82.93,0.05,-1.17
Soil_Pollution_Index,200.0,76.49,39.69,11.15,40.89,78.60,109.21,149.23,68.32,0.03,-1.14
Industrial_Waste (in tons),200.0,52891.68,27224.49,1019.37,31201.97,55299.15,74805.82,99739.36,43603.85,-0.09,-1.03
Energy_Recovered (in GWh),200.0,260.45,147.14,11.73,118.35,273.14,384.96,499.98,266.60,-0.04,-1.27
CO2_Emissions (in MT),200.0,24.88,14.47,1.92,11.22,25.36,38.55,49.69,27.33,0.03,-1.33
Renewable_Energy (%),200.0,27.80,12.36,5.04,17.70,29.17,37.07,49.56,19.37,-0.09,-1.07
Plastic_Waste_Produced (in tons),200.0,24492.89,14421.36,542.95,12843.88,24121.54,36516.23,49852.28,23672.35,0.03,-1.14
Energy_Consumption_Per_Capita (in MWh),200.0,9.44,5.58,0.53,4.58,9.22,14.00,19.98,9.42,0.19,-1.14


**Key Takeaways & Corrected Interpretation of Distributional Metrics:**
- **Pollution Indices:** Air Pollution Index exhibits the widest absolute range (50.30 to 297.95, mean: 180.63), while Water Pollution Index averages 115.07 (31.13 to 199.32), and Soil Pollution Index averages 76.49 (11.15 to 149.23).
- **Target Variable (`Energy_Recovered (in GWh)`):** Spans evenly from 11.73 GWh to 499.98 GWh with a mean of 260.45 GWh and median of 273.56 GWh.
- **Statistical Interpretation of Skewness and Kurtosis:**
  - All numerical features exhibit sample skewness values between $-0.20$ and $+0.25$, indicating **limited asymmetry around the mean**.
  - Crucially, **skewness measures only directional asymmetry, whereas kurtosis is the metric directly relevant to tail weight and outlier propensity**.
  - Therefore, low skewness alone does not prove the absence of heavy tails or that non-linear transformations are inherently unnecessary. Rather, examination of both the sample kurtosis (all between $-1.0$ and $-1.3$, reflecting sub-Gaussian, relatively flat tails) and the bounded empirical ranges indicates that standard z-score standardization is appropriate for linear and logistic modeling without requiring aggressive power transforms.

## 4. Exploratory Data Analysis (EDA)
We construct publication-grade visualizations to uncover patterns, dependencies, temporal trajectories, and target relationships. All figures are persisted to `outputs/visualizations/`.

### 4.1 Correlation Analysis
We compute Pearson correlation coefficients across all numerical features to examine pairwise linear dependencies and check for multicollinearity.

In [8]:
plt.figure(figsize=(11, 8.5))
corr_matrix = df[num_cols].corr()

# Mask upper triangle for visual clarity
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

clean_labels = [c.replace(' (in tons)', '').replace(' (in GWh)', '').replace(' (in MT)', '')
                 .replace(' (in MWh)', '').replace(' (in millions)', '').replace(' (in USD)', '') 
                for c in num_cols]

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-0.3, vmax=0.3, center=0, linewidths=0.5,
            xticklabels=clean_labels, yticklabels=clean_labels,
            cbar_kws={'label': 'Pearson Correlation Coefficient (r)'})

plt.title('Correlation Heatmap of Numerical Features', fontsize=14, weight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/correlation_heatmap.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/3824732429.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Correlation Heatmap:**
1. **Bivariate Target Correlations:** `Energy_Recovered (in GWh)` exhibits weak pairwise linear correlations with all candidate predictors:
   - Industrial Waste: $r = -0.16$ (weak negative linear association)
   - Year: $r = +0.09$
   - Soil Pollution Index: $r = +0.06$
   - Renewable Energy (%): $r = +0.04$
   - CO2 Emissions: $r = +0.02$
   - Air Pollution Index: $r = +0.003$
   - Energy Consumption Per Capita: $r = -0.03$
2. **Absence of Multicollinearity:** No pair of numerical predictors has a correlation coefficient $|r| > 0.30$. The highest observed correlations are $r = +0.24$ (Water Pollution Index and Population) and $r = -0.22$ (Renewable Energy and GDP Per Capita).
3. **Statistical Distinction between Bivariate Association and Multiple Regression:**
   - A weak Pearson correlation coefficient indicates only weak **pairwise linear association** between two individual variables.
   - Importantly, **weak individual bivariate correlations do not rule out combined predictive value in multiple linear regression**, where linear combinations, interaction proxies, or suppressor effects can account for variance that is not apparent in isolated pairwise projections.
   - We therefore treat multiple linear regression as an empirical hypothesis to be rigorously tested against a dummy baseline.

### 4.2 Pollution Distribution & Central Tendency
We compare the distributions and spreads of `Air_Pollution_Index`, `Water_Pollution_Index`, and `Soil_Pollution_Index` using box plots.

In [9]:
plt.figure(figsize=(9, 5.5))
palette = {'Air_Pollution_Index': '#e74c3c', 'Water_Pollution_Index': '#3498db', 'Soil_Pollution_Index': '#2ecc71'}
sns.boxplot(data=df[['Air_Pollution_Index', 'Water_Pollution_Index', 'Soil_Pollution_Index']], 
            palette=palette, width=0.45, flierprops={'marker': 'o', 'markersize': 5, 'markerfacecolor': 'grey'})

plt.title('Distribution Comparison of Pollution Indices', fontsize=14, weight='bold', pad=15)
plt.ylabel('Index Score', fontsize=12)
plt.xticks([0, 1, 2], ['Air Pollution Index', 'Water Pollution Index', 'Soil Pollution Index'], fontsize=11)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/pollution_boxplots.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/1977928728.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Pollution Box Plots:**
- `Air_Pollution_Index` has the highest central tendency (median $\approx 184$, IQR $\approx 103$), spanning from 50 to nearly 300.
- `Water_Pollution_Index` is situated in the intermediate range (median $\approx 113$, IQR $\approx 83$), spanning roughly 30 to 200.
- `Soil_Pollution_Index` is on the lowest scale (median $\approx 79$, IQR $\approx 69$), bounded between 11 and 149.
- No points fall outside $1.5 \times \text{IQR}$, indicating zero statistical outliers in any of the three pollution indices.

### 4.3 Pollution Trends Over Time (2000–2019)
We evaluate the annual trajectories of the three pollution indices across the 20-year observation window.

In [10]:
plt.figure(figsize=(11, 6))
yearly = df.groupby('Year')[['Air_Pollution_Index', 'Water_Pollution_Index', 'Soil_Pollution_Index']].mean().reset_index()

plt.plot(yearly['Year'], yearly['Air_Pollution_Index'], marker='o', linewidth=2.5, label='Air Pollution Index', color='#e74c3c')
plt.plot(yearly['Year'], yearly['Water_Pollution_Index'], marker='s', linewidth=2.5, label='Water Pollution Index', color='#3498db')
plt.plot(yearly['Year'], yearly['Soil_Pollution_Index'], marker='^', linewidth=2.5, label='Soil Pollution Index', color='#2ecc71')

plt.title('Annual Mean Pollution Indices (2000–2019)', fontsize=14, weight='bold', pad=15)
plt.xlabel('Year', fontsize=12, labelpad=10)
plt.ylabel('Mean Pollution Index Score', fontsize=12, labelpad=10)
plt.xticks(yearly['Year'], rotation=45)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/pollution_trends_over_time.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/3208729199.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Pollution Trends Over Time:**
- Annual means oscillate within stable boundaries across all two decades (Air: 155–205; Water: 95–135; Soil: 65–90).
- **Critical Methodological Limitation:** Because each year samples a distinct cross-section of countries (~5 to 15 different nations per year), **yearly aggregates reflect changing country composition and cannot automatically be interpreted as global temporal trends or synchronized environmental waves**. Year functions primarily as a cross-sectional time covariate rather than a continuous longitudinal panel identifier.

### 4.4 Country Pollution Comparison
We examine the 10 lowest and 10 highest countries ranked by their recorded mean Air Pollution Index.

In [11]:
country_air = df.groupby('Country')['Air_Pollution_Index'].mean().sort_values()
top_bottom_countries = pd.concat([country_air.head(10), country_air.tail(10)])

plt.figure(figsize=(12, 7))
colors = ['#2ecc71'] * 10 + ['#e74c3c'] * 10
plt.barh(top_bottom_countries.index, top_bottom_countries.values, color=colors, edgecolor='black', alpha=0.85)

global_mean = df['Air_Pollution_Index'].mean()
plt.axvline(global_mean, color='black', linestyle='--', linewidth=1.5, label=f'Global Mean ({global_mean:.1f})')

plt.title('Country Comparison: Lowest 10 vs. Highest 10 Mean Air Pollution Index', fontsize=14, weight='bold', pad=15)
plt.xlabel('Mean Air Pollution Index', fontsize=12, labelpad=10)
plt.ylabel('Country', fontsize=12)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/country_pollution_comparison.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/3943100148.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Country Comparisons:**
- The lowest recorded air pollution scores (e.g., bottom 10) range between 50 and 80.
- The highest recorded air pollution scores (e.g., top 10) exceed 270.
- **Statistical Caution:** Because 152 of the 175 countries appear only once in the dataset, these country comparisons reflect individual historical sample records rather than long-term longitudinal country baselines. Rankings should not be interpreted as permanent national performance indices.

### 4.5 Industrial and Plastic Waste Distributions
We inspect the distribution, variance, and density of industrial and plastic waste.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Industrial_Waste (in tons)'], kde=True, ax=axes[0], color='#8e44ad', bins=15)
axes[0].set_title('Industrial Waste Distribution', fontsize=13, weight='bold')
axes[0].set_xlabel('Industrial Waste (tons)', fontsize=11)
axes[0].set_ylabel('Observation Count', fontsize=11)

sns.histplot(df['Plastic_Waste_Produced (in tons)'], kde=True, ax=axes[1], color='#16a085', bins=15)
axes[1].set_title('Plastic Waste Produced Distribution', fontsize=13, weight='bold')
axes[1].set_xlabel('Plastic Waste (tons)', fontsize=11)
axes[1].set_ylabel('Observation Count', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/industrial_waste_distribution.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/1747168774.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Waste Distributions:**
- Industrial waste (1,000 to 100,000 tons) and plastic waste (500 to 50,000 tons) exhibit uniform-like dispersion across their operational ranges.
- There are no severe zero-inflation artifacts, clipping boundaries, or extreme long tails.

### 4.6 Energy Recovery Target Distribution & Bivariate Relationships
We examine the distribution of `Energy_Recovered (in GWh)` and its relationships with key industrial and pollution covariates.

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df['Energy_Recovered (in GWh)'], kde=True, ax=axes[0], color='#27ae60', bins=15)
axes[0].set_title('Energy Recovered (GWh) - Histogram & KDE', fontsize=13, weight='bold')
axes[0].set_xlabel('Energy Recovered (GWh)', fontsize=11)
axes[0].set_ylabel('Observation Count', fontsize=11)

sns.boxplot(x=df['Energy_Recovered (in GWh)'], ax=axes[1], color='#a9dfbf', width=0.3)
axes[1].set_title('Energy Recovered (GWh) - Boxplot', fontsize=13, weight='bold')
axes[1].set_xlabel('Energy Recovered (GWh)', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/energy_recovery_distribution.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/1876254973.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

bivariate_features = [
    ('Industrial_Waste (in tons)', 'Industrial Waste (tons)', '#8e44ad'),
    ('CO2_Emissions (in MT)', 'CO2 Emissions (MT)', '#d35400'),
    ('Air_Pollution_Index', 'Air Pollution Index', '#c0392b'),
    ('Energy_Consumption_Per_Capita (in MWh)', 'Energy Consumption Per Capita (MWh)', '#2980b9')
]

for ax, (col, label, clr) in zip(axes.flatten(), bivariate_features):
    sns.regplot(data=df, x=col, y='Energy_Recovered (in GWh)', ax=ax,
                color=clr, scatter_kws={'alpha': 0.6, 'edgecolor': 'k', 's': 45},
                line_kws={'color': 'black', 'linewidth': 1.5, 'linestyle': '--'})
    r_val = df[col].corr(df['Energy_Recovered (in GWh)'])
    ax.set_title(f'Energy Recovered vs. {label}\n(Pearson r = {r_val:.3f})', fontsize=12, weight='bold')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Energy Recovered (GWh)', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/energy_recovery_relationships.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/502355710.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation of Energy Recovery Relationships:**
- `Energy_Recovered (in GWh)` is uniformly distributed between 11.7 GWh and 500 GWh (Mean: 260.5 GWh, Median: 273.6 GWh).
- Fitted OLS regression lines against Industrial Waste ($r = -0.161$), CO2 Emissions ($r = 0.025$), Air Pollution Index ($r = 0.003$), and Per Capita Energy Consumption ($r = -0.030$) demonstrate virtually horizontal slopes.
- This empirical evidence confirms that energy recovery is not linearly determined by individual pollution or waste features alone, setting realistic performance expectations for linear modeling.

## 6. Feature Engineering and Mathematical Derivations

In this section, we construct engineered features with strict physical and mathematical discipline:
1. **Preserve Original Columns:** All original raw features remain intact.
2. **Total Energy Consumption (GWh):** Derived from per-capita energy consumption and population after rigorous unit verification.
3. **Waste-Per-Capita Indicators (kg/person):** Derived from industrial and plastic waste divided by population after checking denominators.
4. **Discipline:** Arbitrary pollution-to-waste ratios are avoided to prevent non-physical artifacts or unstable divisions.
5. **Feature Sets Formulation:** We create two candidate feature spaces ($X_{\text{base}}$ vs. $X_{\text{engineered}}$) to empirically test whether derived features add predictive value during modeling.

### 6.1 Total Energy Consumption: Mathematical Derivation & Unit Verification

#### Physical Formula:
$$\text{Total GWh} = \text{Energy\_Consumption\_Per\_Capita} \left(\frac{\text{MWh}}{\text{person}}\right) \times (\text{Population} \times 10^6 \text{ persons}) \times \left(\frac{1 \text{ GWh}}{1000 \text{ MWh}}\right)$$

Simplifying constants:
$$\text{Energy\_Consumption\_Total\_GWh} = \text{Energy\_Consumption\_Per\_Capita (in MWh)} \times \text{Population (in millions)} \times 1000$$

#### Methodological Note:
Because this feature is a deterministic mathematical product of two existing columns (`Energy_Consumption_Per_Capita` and `Population`), **it does not introduce independent external information**. We include it to evaluate whether a linear model benefits from having national aggregate energy scale explicitly available alongside intensity metrics.

In [15]:
# Derive Total Energy Consumption in GWh
df['Energy_Consumption_Total_GWh'] = (
    df['Energy_Consumption_Per_Capita (in MWh)'] * df['Population (in millions)'] * 1000.0
)

print("Engineered Feature: Energy_Consumption_Total_GWh Summary:")
print(df['Energy_Consumption_Total_GWh'].describe().round(2))

Engineered Feature: Energy_Consumption_Total_GWh Summary:
count        200.00
mean      996661.89
std       846553.10
min         8266.30
25%       280058.10
50%       763275.00
75%      1599265.70
max      3630251.50
Name: Energy_Consumption_Total_GWh, dtype: float64


### 6.2 Waste-Per-Capita Indicators: Denominator Audit & Derivation

#### Denominator Audit:
Before computing per-capita metrics, we verify that `Population (in millions)` contains no zero or near-zero values that could cause numerical instability:
- $\min(\text{Population}) = 2.32\text{ million}$
- $\max(\text{Population}) = 198.82\text{ million}$
- Zero division risk: **None**.

#### Derivation Formulas:
$$\text{Industrial\_Waste\_Per\_Capita\_kg} = \frac{\text{Industrial\_Waste (in tons)} \times 1000 \text{ kg/ton}}{\text{Population (in millions)} \times 10^6 \text{ persons}} = \frac{\text{Industrial\_Waste (in tons)}}{\text{Population (in millions)} \times 1000}$$

$$\text{Plastic\_Waste\_Per\_Capita\_kg} = \frac{\text{Plastic\_Waste\_Produced (in tons)} \times 1000 \text{ kg/ton}}{\text{Population (in millions)} \times 10^6 \text{ persons}} = \frac{\text{Plastic\_Waste\_Produced (in tons)}}{\text{Population (in millions)} \times 1000}$$

In [16]:
# Derive waste per capita in kg/person
df['Industrial_Waste_Per_Capita_kg'] = (df['Industrial_Waste (in tons)'] * 1000.0) / (df['Population (in millions)'] * 1e6)
df['Plastic_Waste_Per_Capita_kg'] = (df['Plastic_Waste_Produced (in tons)'] * 1000.0) / (df['Population (in millions)'] * 1e6)

print("Waste Per Capita Indicators Summary (kg/person):")
df[['Industrial_Waste_Per_Capita_kg', 'Plastic_Waste_Per_Capita_kg']].describe().round(4)

Waste Per Capita Indicators Summary (kg/person):


,Industrial_Waste_Per_Capita_kg,Plastic_Waste_Per_Capita_kg
count,200.0000,200.0000
mean,1.1709,0.5986
std,2.7441,1.6541
min,0.0092,0.0028
25%,0.2902,0.1108
50%,0.5375,0.2233
75%,0.9217,0.4819
max,30.8338,18.7823


### 6.3 Feature Sets Formulation
We formally define the candidate feature lists for regression and classification:
1. **Base Regression Features ($X_{\text{base}}$):** All raw numerical predictors excluding the target (`Energy_Recovered`) and identifier (`Country`).
2. **Engineered Regression Features ($X_{\text{engineered}}$):** Base features plus `Energy_Consumption_Total_GWh`, `Industrial_Waste_Per_Capita_kg`, and `Plastic_Waste_Per_Capita_kg`.
3. **Classification Features ($X_{\text{clf}}$):** All predictors with **`Air_Pollution_Index` strictly excluded** to prevent catastrophic target leakage.

In [17]:
# 1. Base Regression Feature Set
features_reg_base = [
    'Air_Pollution_Index',
    'Water_Pollution_Index',
    'Soil_Pollution_Index',
    'Industrial_Waste (in tons)',
    'CO2_Emissions (in MT)',
    'Renewable_Energy (%)',
    'Plastic_Waste_Produced (in tons)',
    'Energy_Consumption_Per_Capita (in MWh)',
    'Population (in millions)',
    'GDP_Per_Capita (in USD)',
    'Year'
]

# 2. Engineered Regression Feature Set
features_reg_engineered = features_reg_base + [
    'Energy_Consumption_Total_GWh',
    'Industrial_Waste_Per_Capita_kg',
    'Plastic_Waste_Per_Capita_kg'
]

# 3. Classification Feature Set (Strictly excluding Air_Pollution_Index!)
features_clf = [c for c in features_reg_engineered if c != 'Air_Pollution_Index']

print(f"Base Regression Features Count: {len(features_reg_base)}")
print(f"Engineered Regression Features Count: {len(features_reg_engineered)}")
print(f"Classification Features Count (Air_Pollution_Index omitted): {len(features_clf)}")

Base Regression Features Count: 11
Engineered Regression Features Count: 14
Classification Features Count (Air_Pollution_Index omitted): 13


## 7. Train/Test Splitting Strategies and Leakage Prevention

To ensure robust evaluation and test generalization under different paradigms, we prepare two distinct splitting strategies:

1. **Strategy 1: Random 80/20 Split**
   - Standard benchmark partitioning with fixed `random_state=42`.
   - Useful for testing model capability on randomly held-out country-year observations.
   - Note: Random splitting does not measure temporal out-of-sample forecasting capability.

2. **Strategy 2: Chronological Split (Temporal Benchmark)**
   - **Training Set:** Years 2000 to 2014 ($N = 153$, 76.5%).
   - **Testing Set:** Years 2015 to 2019 ($N = 47$, 23.5%).
   - Evaluates whether relationships established on historical observations generalize forward in time to future unseen years.

### Strict Leakage Protocol:
- **Tertile Cutoffs:** Quantile cutoffs ($Q_{33.3\%}$ and $Q_{66.7\%}$) for `Pollution_Severity` are calculated **strictly on training observations only**, and then applied to classify test observations.
- **Relative Tertile Classes Note:** These Low, Medium, and High categories represent empirical tertiles calculated relative to the observed distribution of this specific dataset. **They do not correspond to official statutory air quality standards (such as EPA or WHO AQI break-points).**
- **Preprocessing:** All scaling (`StandardScaler`) must be fitted strictly inside training sets (or using `sklearn.pipeline.Pipeline`).

In [18]:
from sklearn.model_selection import train_test_split

# ==============================================================================
# STRATEGY 1: RANDOM 80/20 SPLIT
# ==============================================================================
train_df_rand, test_df_rand = train_test_split(df, test_size=0.20, random_state=42)

# Compute tertiles STRICTLY on train_df_rand
q1_rand = train_df_rand['Air_Pollution_Index'].quantile(1.0 / 3.0)
q2_rand = train_df_rand['Air_Pollution_Index'].quantile(2.0 / 3.0)

def assign_severity(val, q1, q2):
    if val <= q1:
        return 'Low'
    elif val <= q2:
        return 'Medium'
    else:
        return 'High'

# Assign classification target
train_df_rand = train_df_rand.copy()
test_df_rand = test_df_rand.copy()
train_df_rand['Pollution_Severity'] = train_df_rand['Air_Pollution_Index'].apply(lambda x: assign_severity(x, q1_rand, q2_rand))
test_df_rand['Pollution_Severity'] = test_df_rand['Air_Pollution_Index'].apply(lambda x: assign_severity(x, q1_rand, q2_rand))

print("=== STRATEGY 1: RANDOM 80/20 SPLIT ===")
print(f"Train Sample Count: {len(train_df_rand)} ({(len(train_df_rand)/len(df))*100:.1f}%)")
print(f"Test Sample Count:  {len(test_df_rand)} ({(len(test_df_rand)/len(df))*100:.1f}%)")
print(f"Training-Only Air Pollution Cutoffs: Low <= {q1_rand:.2f} < Medium <= {q2_rand:.2f} < High")
print("\nTrain Class Balance:")
print(train_df_rand['Pollution_Severity'].value_counts(normalize=True).round(3) * 100)
print("\nTest Class Balance:")
print(test_df_rand['Pollution_Severity'].value_counts(normalize=True).round(3) * 100)

=== STRATEGY 1: RANDOM 80/20 SPLIT ===
Train Sample Count: 160 (80.0%)
Test Sample Count:  40 (20.0%)
Training-Only Air Pollution Cutoffs: Low <= 151.36 < Medium <= 221.09 < High

Train Class Balance:
Pollution_Severity
Low       33.8
Medium    33.1
High      33.1
Name: proportion, dtype: float64

Test Class Balance:
Pollution_Severity
Low       40.0
Medium    37.5
High      22.5
Name: proportion, dtype: float64


In [19]:
# ==============================================================================
# STRATEGY 2: CHRONOLOGICAL SPLIT (2000-2014 vs. 2015-2019)
# ==============================================================================
train_df_chrono = df[df['Year'] <= 2014].copy()
test_df_chrono = df[df['Year'] >= 2015].copy()

# Compute tertiles STRICTLY on train_df_chrono
q1_chrono = train_df_chrono['Air_Pollution_Index'].quantile(1.0 / 3.0)
q2_chrono = train_df_chrono['Air_Pollution_Index'].quantile(2.0 / 3.0)

train_df_chrono['Pollution_Severity'] = train_df_chrono['Air_Pollution_Index'].apply(lambda x: assign_severity(x, q1_chrono, q2_chrono))
test_df_chrono['Pollution_Severity'] = test_df_chrono['Air_Pollution_Index'].apply(lambda x: assign_severity(x, q1_chrono, q2_chrono))

print("=== STRATEGY 2: CHRONOLOGICAL SPLIT ===")
print(f"Train Sample Count (2000-2014): {len(train_df_chrono)} ({(len(train_df_chrono)/len(df))*100:.1f}%)")
print(f"Test Sample Count (2015-2019):  {len(test_df_chrono)} ({(len(test_df_chrono)/len(df))*100:.1f}%)")
print(f"Training-Only Air Pollution Cutoffs: Low <= {q1_chrono:.2f} < Medium <= {q2_chrono:.2f} < High")
print("\nTrain Class Balance:")
print(train_df_chrono['Pollution_Severity'].value_counts(normalize=True).round(3) * 100)
print("\nTest Class Balance:")
print(test_df_chrono['Pollution_Severity'].value_counts(normalize=True).round(3) * 100)

=== STRATEGY 2: CHRONOLOGICAL SPLIT ===
Train Sample Count (2000-2014): 153 (76.5%)
Test Sample Count (2015-2019):  47 (23.5%)
Training-Only Air Pollution Cutoffs: Low <= 150.06 < Medium <= 219.89 < High

Train Class Balance:
Pollution_Severity
High      33.3
Low       33.3
Medium    33.3
Name: proportion, dtype: float64

Test Class Balance:
Pollution_Severity
Medium    40.4
Low       31.9
High      27.7
Name: proportion, dtype: float64


### 7.3 Target Distributions Across Splits
We inspect the regression target `Energy_Recovered (in GWh)` across the training and testing partitions for both splitting strategies.

In [20]:
target_col = 'Energy_Recovered (in GWh)'

target_dist_summary = pd.DataFrame({
    'Random_Train': train_df_rand[target_col].describe(),
    'Random_Test': test_df_rand[target_col].describe(),
    'Chrono_Train': train_df_chrono[target_col].describe(),
    'Chrono_Test': test_df_chrono[target_col].describe()
}).round(2)

print("=== Target (Energy_Recovered in GWh) Distribution Across Splits ===")
target_dist_summary

=== Target (Energy_Recovered in GWh) Distribution Across Splits ===


,Random_Train,Random_Test,Chrono_Train,Chrono_Test
count,160.00,40.00,153.00,47.00
mean,265.73,239.32,250.69,292.23
std,144.48,157.49,149.02,137.61
min,11.73,13.22,11.73,15.98
25%,142.95,92.86,100.57,164.88
50%,283.22,237.24,253.84,312.30
75%,385.53,375.10,371.79,399.48
max,499.98,493.12,499.98,492.08


### 7.4 Comprehensive Anti-Leakage Verification Checks
Before proceeding to modeling, we run automated assertions to confirm that:
1. `Air_Pollution_Index` is completely absent from classification predictors ($X_{\text{clf}}$).
2. The regression target `Energy_Recovered (in GWh)` is absent from both feature matrices.
3. No overlap exists between training and testing indices in either split.
4. Classification categories contain zero empty classes.

In [21]:
# 1. Target Leakage Checks
assert 'Air_Pollution_Index' not in features_clf, "LEAKAGE ALERT: Air_Pollution_Index detected in classification features!"
assert 'Energy_Recovered (in GWh)' not in features_reg_base, "LEAKAGE ALERT: Target detected in base regression features!"
assert 'Energy_Recovered (in GWh)' not in features_reg_engineered, "LEAKAGE ALERT: Target detected in engineered regression features!"
assert 'Energy_Recovered (in GWh)' not in features_clf, "LEAKAGE ALERT: Target detected in classification features!"

# 2. Partition Independence Checks
assert len(set(train_df_rand.index).intersection(set(test_df_rand.index))) == 0, "Index overlap detected in Random split!"
assert len(set(train_df_chrono.index).intersection(set(test_df_chrono.index))) == 0, "Index overlap detected in Chronological split!"

# 3. Non-empty Class Checks
assert len(train_df_rand['Pollution_Severity'].unique()) == 3, "Missing classes in Random Train!"
assert len(test_df_rand['Pollution_Severity'].unique()) == 3, "Missing classes in Random Test!"
assert len(train_df_chrono['Pollution_Severity'].unique()) == 3, "Missing classes in Chrono Train!"
assert len(test_df_chrono['Pollution_Severity'].unique()) == 3, "Missing classes in Chrono Test!"

print("ALL LEAKAGE & INTEGRITY CHECKS PASSED SUCCESSFULLY!")
print("1. Target columns strictly excluded from feature sets.")
print("2. Classification feature set X_clf verified free of Air_Pollution_Index.")
print("3. Split partitions verified disjoint.")
print("4. All multiclass categories verified non-empty.")

ALL LEAKAGE & INTEGRITY CHECKS PASSED SUCCESSFULLY!
1. Target columns strictly excluded from feature sets.
2. Classification feature set X_clf verified free of Air_Pollution_Index.
3. Split partitions verified disjoint.
4. All multiclass categories verified non-empty.


## 8. Predictive Modeling: Linear Regression for Energy Recovery

### Objective
Predict `Energy_Recovered (in GWh)` using industrial and pollution covariates.

### Models Evaluated:
1. **DummyRegressor (Baseline):** Predicts the training set mean (`strategy='mean'`). This represents the null hypothesis that feature relationships do not generalize beyond predicting the central tendency.
2. **Linear Regression (Base Features):** Standard OLS trained on the 11 original continuous features with `StandardScaler` fitted strictly on training data.
3. **Linear Regression (Engineered Features):** Standard OLS trained on all 14 features (including `Energy_Consumption_Total_GWh`, `Industrial_Waste_Per_Capita_kg`, and `Plastic_Waste_Per_Capita_kg`).

### Evaluation Protocols:
Both models are evaluated on:
- Strategy 1: Random 80/20 Split
- Strategy 2: Chronological Split (Train: 2000–2014, Test: 2015–2019)

### Evaluation Metrics:
- **MAE (Mean Absolute Error):** Average magnitude of errors in GWh.
- **MSE (Mean Squared Error):** Average squared error penalty.
- **RMSE (Root Mean Squared Error):** Standard deviation of unexplained residuals in GWh.
- **$R^2$ (Coefficient of Determination):** Proportion of variance explained relative to the mean. Note that out-of-sample $R^2$ can be negative if a model's sum of squared errors exceeds the sum of squared errors of predicting the mean.

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper function to compute regression metrics
def compute_reg_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Function to run complete regression evaluation
def run_regression_suite(train_df, test_df, split_name):
    y_train = train_df[target_col]
    y_test = test_df[target_col]
    
    # 1. Baseline: Dummy Regressor (Train Mean)
    dummy = DummyRegressor(strategy='mean')
    dummy.fit(train_df[features_reg_base], y_train)
    y_pred_dummy = dummy.predict(test_df[features_reg_base])
    metrics_dummy = compute_reg_metrics(y_test, y_pred_dummy)
    
    # 2. Linear Regression (Base Features)
    pipe_base = Pipeline([
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    pipe_base.fit(train_df[features_reg_base], y_train)
    y_pred_base = pipe_base.predict(test_df[features_reg_base])
    metrics_base = compute_reg_metrics(y_test, y_pred_base)
    
    # 3. Linear Regression (Engineered Features)
    pipe_eng = Pipeline([
        ('scaler', StandardScaler()),
        ('reg', LinearRegression())
    ])
    pipe_eng.fit(train_df[features_reg_engineered], y_train)
    y_pred_eng = pipe_eng.predict(test_df[features_reg_engineered])
    metrics_eng = compute_reg_metrics(y_test, y_pred_eng)
    
    results = pd.DataFrame([metrics_dummy, metrics_base, metrics_eng],
                           index=['Dummy Baseline (Mean)', 'Linear Reg (Base Features)', 'Linear Reg (Engineered Features)'])
    results.columns.name = split_name
    
    preds_dict = {
        'y_test': y_test,
        'y_pred_dummy': y_pred_dummy,
        'y_pred_base': y_pred_base,
        'y_pred_eng': y_pred_eng,
        'pipe_base': pipe_base,
        'pipe_eng': pipe_eng
    }
    return results, preds_dict

reg_results_rand, preds_rand = run_regression_suite(train_df_rand, test_df_rand, 'Random Split (80/20)')
reg_results_chrono, preds_chrono = run_regression_suite(train_df_chrono, test_df_chrono, 'Chronological Split (2000-2014 vs 2015-2019)')

print("=== REGRESSION RESULTS: STRATEGY 1 (RANDOM 80/20 SPLIT) ===")
display(reg_results_rand.round(3))

print("\n=== REGRESSION RESULTS: STRATEGY 2 (CHRONOLOGICAL SPLIT) ===")
display(reg_results_chrono.round(3))

=== REGRESSION RESULTS: STRATEGY 1 (RANDOM 80/20 SPLIT) ===


Random Split (80/20),MAE,MSE,RMSE,R2
Dummy Baseline (Mean),141.649,24879.622,157.733,-0.029
Linear Reg (Base Features),145.979,27715.079,166.478,-0.146
Linear Reg (Engineered Features),150.251,29986.718,173.167,-0.240



=== REGRESSION RESULTS: STRATEGY 2 (CHRONOLOGICAL SPLIT) ===


Chronological Split (2000-2014 vs 2015-2019),MAE,MSE,RMSE,R2
Dummy Baseline (Mean),125.363,20259.570,142.336,-0.093
Linear Reg (Base Features),121.412,18918.328,137.544,-0.021
Linear Reg (Engineered Features),120.528,18761.323,136.972,-0.012


### 8.3 Regression Diagnostic Visualizations
We generate diagnostic plots for both splits:
1. **Actual vs. Predicted Plots:** Compares predicted GWh values against ground truth along the 45-degree identity line ($y = x$).
2. **Residual Plots:** Examines residual errors ($y - \hat{y}$) against predicted values and their error distributions to inspect heteroscedasticity or systematic bias.

In [23]:
# 1. Actual vs. Predicted Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Random Split
ax = axes[0]
y_te_r = preds_rand['y_test']
y_pr_b_r = preds_rand['y_pred_base']
y_pr_e_r = preds_rand['y_pred_eng']

ax.scatter(y_te_r, y_pr_b_r, color='#2980b9', alpha=0.7, edgecolors='k', s=50, label='Base Features (11)')
ax.scatter(y_te_r, y_pr_e_r, color='#e67e22', marker='^', alpha=0.7, edgecolors='k', s=50, label='Engineered Features (14)')
ax.plot([0, 500], [0, 500], 'r--', linewidth=2, label='Perfect Prediction (y = x)')
ax.axhline(train_df_rand[target_col].mean(), color='grey', linestyle=':', label='Train Mean Baseline')
ax.set_title('Random 80/20 Split: Actual vs. Predicted', fontsize=13, weight='bold')
ax.set_xlabel('Actual Energy Recovered (GWh)', fontsize=11)
ax.set_ylabel('Predicted Energy Recovered (GWh)', fontsize=11)
ax.set_xlim(0, 520)
ax.set_ylim(0, 520)
ax.legend(loc='upper left', frameon=True)

# Chronological Split
ax = axes[1]
y_te_c = preds_chrono['y_test']
y_pr_b_c = preds_chrono['y_pred_base']
y_pr_e_c = preds_chrono['y_pred_eng']

ax.scatter(y_te_c, y_pr_b_c, color='#2980b9', alpha=0.7, edgecolors='k', s=50, label='Base Features (11)')
ax.scatter(y_te_c, y_pr_e_c, color='#e67e22', marker='^', alpha=0.7, edgecolors='k', s=50, label='Engineered Features (14)')
ax.plot([0, 500], [0, 500], 'r--', linewidth=2, label='Perfect Prediction (y = x)')
ax.axhline(train_df_chrono[target_col].mean(), color='grey', linestyle=':', label='Train Mean Baseline')
ax.set_title('Chronological Split: Actual vs. Predicted', fontsize=13, weight='bold')
ax.set_xlabel('Actual Energy Recovered (GWh)', fontsize=11)
ax.set_ylabel('Predicted Energy Recovered (GWh)', fontsize=11)
ax.set_xlim(0, 520)
ax.set_ylim(0, 520)
ax.legend(loc='upper left', frameon=True)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/regression_actual_vs_predicted.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/2907390518.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# 2. Residual Diagnostic Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Random Split Residuals vs Predicted
res_r_b = y_te_r - y_pr_b_r
res_r_e = y_te_r - y_pr_e_r

axes[0, 0].scatter(y_pr_b_r, res_r_b, color='#2980b9', alpha=0.7, edgecolors='k', s=45, label='Base Features')
axes[0, 0].scatter(y_pr_e_r, res_r_e, color='#e67e22', marker='^', alpha=0.7, edgecolors='k', s=45, label='Engineered Features')
axes[0, 0].axhline(0, color='r', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('Random Split: Residuals vs. Predicted Values', fontsize=12, weight='bold')
axes[0, 0].set_xlabel('Predicted Energy Recovered (GWh)', fontsize=10)
axes[0, 0].set_ylabel('Residual Error (Actual - Pred)', fontsize=10)
axes[0, 0].legend()

# Random Split Residual Distribution
sns.histplot(res_r_b, kde=True, ax=axes[0, 1], color='#2980b9', label='Base Features', alpha=0.5, bins=12)
sns.histplot(res_r_e, kde=True, ax=axes[0, 1], color='#e67e22', label='Engineered Features', alpha=0.5, bins=12)
axes[0, 1].axvline(0, color='r', linestyle='--', linewidth=1.5)
axes[0, 1].set_title('Random Split: Residual Error Distribution', fontsize=12, weight='bold')
axes[0, 1].set_xlabel('Residual Error (GWh)', fontsize=10)
axes[0, 1].legend()

# Chrono Split Residuals vs Predicted
res_c_b = y_te_c - y_pr_b_c
res_c_e = y_te_c - y_pr_e_c

axes[1, 0].scatter(y_pr_b_c, res_c_b, color='#2980b9', alpha=0.7, edgecolors='k', s=45, label='Base Features')
axes[1, 0].scatter(y_pr_e_c, res_c_e, color='#e67e22', marker='^', alpha=0.7, edgecolors='k', s=45, label='Engineered Features')
axes[1, 0].axhline(0, color='r', linestyle='--', linewidth=1.5)
axes[1, 0].set_title('Chrono Split: Residuals vs. Predicted Values', fontsize=12, weight='bold')
axes[1, 0].set_xlabel('Predicted Energy Recovered (GWh)', fontsize=10)
axes[1, 0].set_ylabel('Residual Error (Actual - Pred)', fontsize=10)
axes[1, 0].legend()

# Chrono Split Residual Distribution
sns.histplot(res_c_b, kde=True, ax=axes[1, 1], color='#2980b9', label='Base Features', alpha=0.5, bins=12)
sns.histplot(res_c_e, kde=True, ax=axes[1, 1], color='#e67e22', label='Engineered Features', alpha=0.5, bins=12)
axes[1, 1].axvline(0, color='r', linestyle='--', linewidth=1.5)
axes[1, 1].set_title('Chrono Split: Residual Error Distribution', fontsize=12, weight='bold')
axes[1, 1].set_xlabel('Residual Error (GWh)', fontsize=10)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/regression_residuals.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/1641585749.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.4 Honest Interpretation of Regression Results

#### 1. Baseline Comparison & Negative $R^2$:
- In the **Random Split**:
  - The Dummy Baseline achieves $\text{MAE} = 141.65\text{ GWh}$, $\text{RMSE} = 157.73\text{ GWh}$, and $R^2 = -0.029$.
  - Linear Regression with Base Features yields $\text{MAE} = 145.98\text{ GWh}$, $\text{RMSE} = 166.48\text{ GWh}$, and $R^2 = -0.146$.
  - Linear Regression with Engineered Features yields $\text{MAE} = 150.25\text{ GWh}$, $\text{RMSE} = 173.17\text{ GWh}$, and $R^2 = -0.240$.
  - **Critical Finding:** Both linear models perform **worse than the simple dummy mean baseline** on held-out test data. An out-of-sample $R^2 < 0$ demonstrates that the model's residual sum of squares exceeds the total sum of squares around the mean.
- In the **Chronological Split**:
  - The Dummy Baseline yields $R^2 = -0.093$ because the test set (2015–2019) has a higher empirical mean ($292.23\text{ GWh}$) than the training set ($250.69\text{ GWh}$).
  - Linear Regression achieves $R^2 = -0.021$ (Base) and $R^2 = -0.012$ (Engineered). While the model slightly tracks the positive secular slope of `Year` to reduce error relative to the static training mean, its $R^2$ remains negative ($R^2 < 0$).

#### 2. Comparison of Base vs. Engineered Features:
- Adding the derived features (`Energy_Consumption_Total_GWh`, `Industrial_Waste_Per_Capita_kg`, and `Plastic_Waste_Per_Capita_kg`) **did not provide genuine predictive gain**.
- In the Random split, adding collinear/derived features increased parameter variance, resulting in higher test error (RMSE increased from $166.48$ to $173.17$).
- This confirms our Phase 2 methodological note: deterministic combinations of existing columns cannot fabricate independent explanatory signal where no underlying physical linear association exists.

#### 3. Data Science Conclusion:
Linear combinations of global pollution indices, waste metrics, and macroeconomic aggregates in this dataset cannot reliably predict waste-to-energy recovery in GWh. Transparently documenting this null linear result demonstrates rigorous data science integrity.

## 9. Predictive Modeling: Logistic Regression for Pollution Severity

### Objective
Classify environmental pollution severity into **Low**, **Medium**, and **High** categories using macroeconomic and industrial features.

### Target Definition & Anti-Leakage Protocol:
- As established in Phase 2, `Pollution_Severity` is constructed using empirical tertiles ($Q_{33.3\%}$ and $Q_{66.7\%}$) calculated **strictly on training observations only**.
- **Relative Tertile Classes Note:** These classes represent empirical tertile brackets relative to this dataset's sample distribution. They do not represent regulatory, statutory, or medical air quality thresholds.
- **Strict Leakage Prevention:** `Air_Pollution_Index` is **strictly excluded** from all classification predictor matrices ($X_{\text{clf}}$). The model must rely entirely on auxiliary variables (waste production, emissions, energy consumption, population, GDP, and soil/water pollution).

### Models Evaluated:
1. **DummyClassifier (Baseline):** Predicts the most frequent training category (`strategy='most_frequent'`).
2. **Multiclass Logistic Regression:** Trained using `sklearn.pipeline.Pipeline` with `StandardScaler` fitted strictly on training data (`solver='lbfgs'`, `max_iter=1000`, `random_state=42`).

### Evaluation Metrics:
- Accuracy
- Macro-averaged Precision, Recall, and F1-score (evaluates performance equally across all three classes)
- Multiclass ROC-AUC (computed via One-vs-Rest `multi_class='ovr'` with probability predictions)
- Confusion Matrix

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

def run_classification_suite(train_df, test_df, split_name):
    y_train = train_df['Pollution_Severity']
    y_test = test_df['Pollution_Severity']
    X_train = train_df[features_clf]
    X_test = test_df[features_clf]
    
    classes_order = ['Low', 'Medium', 'High']
    
    # 1. Baseline: Dummy Classifier (Most Frequent)
    dummy_clf = DummyClassifier(strategy='most_frequent')
    dummy_clf.fit(X_train, y_train)
    y_pred_dummy = dummy_clf.predict(X_test)
    y_prob_dummy = dummy_clf.predict_proba(X_test)
    
    # 2. Logistic Regression Pipeline
    pipe_clf = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ])
    pipe_clf.fit(X_train, y_train)
    y_pred_clf = pipe_clf.predict(X_test)
    y_prob_clf = pipe_clf.predict_proba(X_test)
    
    # Metrics calculation helper
    def calc_clf_metrics(y_true, y_pred, y_prob, model_classes):
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro', labels=model_classes)
        except Exception:
            auc = np.nan
        return {'Accuracy': acc, 'Macro Precision': prec, 'Macro Recall': rec, 'Macro F1': f1, 'ROC-AUC (OVR)': auc}
    
    metrics_dummy = calc_clf_metrics(y_test, y_pred_dummy, y_prob_dummy, dummy_clf.classes_)
    metrics_clf = calc_clf_metrics(y_test, y_pred_clf, y_prob_clf, pipe_clf.classes_)
    
    results = pd.DataFrame([metrics_dummy, metrics_clf],
                           index=['Dummy Baseline (Most Frequent)', 'Multiclass Logistic Regression'])
    results.columns.name = split_name
    
    cm_dummy = confusion_matrix(y_test, y_pred_dummy, labels=classes_order)
    cm_clf = confusion_matrix(y_test, y_pred_clf, labels=classes_order)
    
    clf_artifacts = {
        'y_test': y_test,
        'y_pred_clf': y_pred_clf,
        'cm_clf': cm_clf,
        'cm_dummy': cm_dummy,
        'pipe_clf': pipe_clf,
        'classes_order': classes_order
    }
    return results, clf_artifacts

clf_results_rand, clf_artifacts_rand = run_classification_suite(train_df_rand, test_df_rand, 'Random Split (80/20)')
clf_results_chrono, clf_artifacts_chrono = run_classification_suite(train_df_chrono, test_df_chrono, 'Chronological Split (2000-2014 vs 2015-2019)')

print("=== CLASSIFICATION RESULTS: STRATEGY 1 (RANDOM 80/20 SPLIT) ===")
display(clf_results_rand.round(3))

print("\n=== CLASSIFICATION RESULTS: STRATEGY 2 (CHRONOLOGICAL SPLIT) ===")
display(clf_results_chrono.round(3))

=== CLASSIFICATION RESULTS: STRATEGY 1 (RANDOM 80/20 SPLIT) ===


Random Split (80/20),Accuracy,Macro Precision,Macro Recall,Macro F1,ROC-AUC (OVR)
Dummy Baseline (Most Frequent),0.40,0.133,0.333,0.190,0.500
Multiclass Logistic Regression,0.35,0.269,0.306,0.282,0.458



=== CLASSIFICATION RESULTS: STRATEGY 2 (CHRONOLOGICAL SPLIT) ===


Chronological Split (2000-2014 vs 2015-2019),Accuracy,Macro Precision,Macro Recall,Macro F1,ROC-AUC (OVR)
Dummy Baseline (Most Frequent),0.277,0.092,0.333,0.144,0.500
Multiclass Logistic Regression,0.340,0.317,0.314,0.304,0.497


In [26]:
# Display Classification Reports
classes_order = ['Low', 'Medium', 'High']

print("=== CLASSIFICATION REPORT: RANDOM 80/20 SPLIT (LOGISTIC REGRESSION) ===")
print(classification_report(clf_artifacts_rand['y_test'], clf_artifacts_rand['y_pred_clf'], labels=classes_order, zero_division=0))

print("\n=== CLASSIFICATION REPORT: CHRONOLOGICAL SPLIT (LOGISTIC REGRESSION) ===")
print(classification_report(clf_artifacts_chrono['y_test'], clf_artifacts_chrono['y_pred_clf'], labels=classes_order, zero_division=0))

=== CLASSIFICATION REPORT: RANDOM 80/20 SPLIT (LOGISTIC REGRESSION) ===
              precision    recall  f1-score   support

         Low       0.31      0.25      0.28        16
      Medium       0.50      0.67      0.57        15
        High       0.00      0.00      0.00         9

    accuracy                           0.35        40
   macro avg       0.27      0.31      0.28        40
weighted avg       0.31      0.35      0.32        40


=== CLASSIFICATION REPORT: CHRONOLOGICAL SPLIT (LOGISTIC REGRESSION) ===
              precision    recall  f1-score   support

         Low       0.29      0.13      0.18        15
      Medium       0.50      0.58      0.54        19
        High       0.17      0.23      0.19        13

    accuracy                           0.34        47
   macro avg       0.32      0.31      0.30        47
weighted avg       0.34      0.34      0.33        47



### 9.3 Confusion Matrix Visualizations
We plot annotated confusion matrices for both evaluation strategies to inspect class-specific misclassifications.

In [27]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Random Split Confusion Matrix
sns.heatmap(clf_artifacts_rand['cm_clf'], annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=classes_order, yticklabels=classes_order, ax=axes[0],
            annot_kws={'size': 13, 'weight': 'bold'})
axes[0].set_title('Random 80/20 Split: Confusion Matrix\n(Logistic Regression)', fontsize=13, weight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)

# Chronological Split Confusion Matrix
sns.heatmap(clf_artifacts_chrono['cm_clf'], annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=classes_order, yticklabels=classes_order, ax=axes[1],
            annot_kws={'size': 13, 'weight': 'bold'})
axes[1].set_title('Chronological Split: Confusion Matrix\n(Logistic Regression)', fontsize=13, weight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/classification_confusion_matrices.png', dpi=300)
plt.show()

/var/folders/rq/9dtg1ypn7gnb8c1mm2f8xq680000gn/T/ipykernel_60246/119835405.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 9.4 Honest Interpretation of Classification Results

#### 1. Performance Against Baseline:
- In a 3-class balanced classification problem, random guessing provides a nominal reference accuracy of $33.3\%$.
- In the **Random Split**:
  - The Dummy Baseline achieves an accuracy of $40.0\%$ (by always predicting 'Low', which comprised $16/40$ test records), with Macro F1 of $0.190$.
  - Logistic Regression achieves an accuracy of **$35.0\%$**, Macro Precision of **$0.269$**, Macro Recall of **$0.306$**, Macro F1 of **$0.282$**, and an One-vs-Rest ROC-AUC of **$0.458$**.
- In the **Chronological Split**:
  - The Dummy Baseline achieves an accuracy of $27.7\%$, with Macro F1 of $0.144$.
  - Logistic Regression achieves an accuracy of **$34.0\%$**, Macro Precision of **$0.317$**, Macro Recall of **$0.314$**, Macro F1 of **$0.304$**, and an One-vs-Rest ROC-AUC of **$0.497$**.

#### 2. Class Imbalance and Distribution Analysis:
- In the training partitions, classes are balanced ($33.3\%$ in each category by tertile definition).
- In the test partitions, sampling variations occur (Random test: 16 Low, 15 Medium, 9 High; Chrono test: 15 Low, 19 Medium, 13 High).
- In the Random split, Logistic Regression struggled on the minority High class (precision = 0.00, recall = 0.00), predicting only Low and Medium.
- In the Chronological split, predictions were distributed across all three classes, exhibiting weak classification performance with an accuracy of $34.0\%$ and an One-vs-Rest ROC-AUC near 0.5 ($0.497$).

#### 3. Methodological Significance & Target Leakage Prevention:
- **Why is performance modest?**
  Because our experimental protocol **strictly excluded `Air_Pollution_Index`** from the predictors to prevent target leakage. Auxiliary indicators (such as GDP per capita, population, and plastic waste) have weak pairwise linear associations with air pollution severity in this dataset.
- Had we included `Air_Pollution_Index` or a direct mathematical transformation of it in $X_{\text{clf}}$, the model would have reported an artificial $100\%$ accuracy through target leakage.
- Demonstrating weak classification performance under strict anti-leakage rules shows that ambient air pollution severity cannot be reliably inferred from these macroeconomic and waste aggregates alone using a linear decision boundary.

## 10. Analytical Insights, Recommendations, and Conclusion

In this final section, we synthesize the empirical findings from our exploratory data analysis, regression modeling, and multiclass classification. We strictly distinguish between direct observations from the dataset, model-derived findings, and analytical interpretations/hypotheses. We also provide evidence-based recommendations, outline core limitations, and propose future directions.

---

### 10.1 Analytical Insights

#### 1. Major Patterns Observed in Air, Water, and Soil Pollution
* **Direct Observations from the Dataset:**
  * The three pollution indicators operate on distinct numerical scales: `Air_Pollution_Index` has the highest central tendency and spread (mean: $180.63$, IQR: $103.11$, range: $50.30 - 297.95$), `Water_Pollution_Index` is intermediate (mean: $115.07$, IQR: $82.85$, range: $31.13 - 199.32$), and `Soil_Pollution_Index` has the lowest scale (mean: $76.49$, IQR: $69.16$, range: $11.15 - 149.23$).
  * Annual aggregate means from 2000 to 2019 oscillate within narrow horizontal bands (Air: $155 - 205$; Water: $95 - 135$; Soil: $65 - 90$) with no evident upward or downward secular global trend.
  * Pairwise linear correlations among the three indices are low: $r = +0.05$ (Air vs. Water), $r = +0.08$ (Air vs. Soil), and $r = +0.01$ (Water vs. Soil).
* **Model-Derived Findings:**
  * In multiple linear regression for energy recovery, standardized coefficients for all three pollution indices were near zero ($|\beta| < 0.08$), providing negligible linear explanatory contribution to energy recovery.
* **Interpretations and Hypotheses:**
  * The low correlations among the three pollution media indicate that in this dataset, air, water, and soil pollution exhibit weak pairwise linear associations rather than behaving as a single synchronized environmental degradation factor.
  * The flat 20-year annual trajectory is an artifact of cross-sectional pooling (sampling ~5 to 15 different nations each year) rather than a synchronized global environmental steady-state.

#### 2. Relationships Between Pollution Indicators and Energy Recovery
* **Direct Observations from the Dataset:**
  * Bivariate Pearson correlations between `Energy_Recovered (in GWh)` and pollution metrics are weak: Air Pollution ($r = +0.003$), Water Pollution ($r = -0.042$), Soil Pollution ($r = +0.064$), and CO2 Emissions ($r = +0.025$).
  * The strongest bivariate correlation with energy recovery is with `Industrial_Waste (in tons)` ($r = -0.161$), representing a weak negative association.
  * Scatter plots with OLS trendlines display essentially horizontal slopes across all pollution variables.
* **Model-Derived Findings:**
  * Both linear regression models produced negative test $R^2$ values (Random split: $R^2 = -0.146$; Chronological split: $R^2 = -0.021$).
* **Interpretations and Hypotheses:**
  * Macro-level ambient pollution indices do not serve as direct proxies for municipal or industrial waste-to-energy recovery.
  * Waste-to-energy capacity is governed by local infrastructure (incineration plants, anaerobic digesters, grid interconnects) and economic subsidies, which are not captured by ambient atmospheric or soil readings.

#### 3. Impact of Feature Engineering on Energy Recovery Prediction
* **Direct Observations from the Dataset:**
  * Derived features included total energy scale (`Energy_Consumption_Total_GWh = Energy_Consumption_Per_Capita * Population * 1000`) and per-capita waste indicators (`Industrial_Waste_Per_Capita_kg`, `Plastic_Waste_Per_Capita_kg`).
  * `Energy_Consumption_Total_GWh` correlates at $r = -0.075$ with energy recovery, while `Industrial_Waste_Per_Capita_kg` correlates at $r = -0.106$.
* **Model-Derived Findings:**
  * In the Random Split, adding engineered features **worsened test error**: RMSE increased from $166.48\text{ GWh}$ to $173.17\text{ GWh}$, and $R^2$ dropped from $-0.146$ to $-0.240$.
  * In the Chronological Split, engineered features shifted RMSE by a negligible $0.57\text{ GWh}$ ($137.54$ vs. $136.97$), with test $R^2$ remaining negative ($-0.012$).
* **Interpretations and Hypotheses:**
  * Because `Energy_Consumption_Total_GWh` is a deterministic product of per-capita energy consumption and population, it introduced collinear scale variance without introducing independent physical signal.
  * Mathematical recombinations of existing features cannot manufacture predictive signal when underlying structural associations with the target are absent.

#### 4. Comparison of Random vs. Chronological Evaluation
* **Direct Observations from the Dataset:**
  * The Chronological test set (2015–2019) exhibits a higher mean energy recovery ($292.23\text{ GWh}$) than the training set ($250.69\text{ GWh}$), reflecting an empirical $+41.54\text{ GWh}$ temporal shift.
  * In contrast, the Random split partitions observations uniformly across time (Train Mean: $265.73\text{ GWh}$, Test Mean: $239.32\text{ GWh}$).
* **Model-Derived Findings:**
  * In the **Random split**, both linear regression models **underperformed the DummyRegressor** (Dummy: $\text{MAE}=141.65$, $\text{RMSE}=157.73$, $R^2=-0.029$; Base Model: $\text{MAE}=145.98$, $\text{RMSE}=166.48$, $R^2=-0.146$).
  * In the **Chronological split**, both linear regression models **slightly outperformed the DummyRegressor on MAE and RMSE** (Dummy: $\text{MAE}=125.36$, $\text{RMSE}=142.34$; Base Model: $\text{MAE}=121.41$, $\text{RMSE}=137.54$; Engineered Model: $\text{MAE}=120.53$, $\text{RMSE}=136.97$), but **retained negative $R^2$ values** (Base: $R^2 = -0.021$; Engineered: $R^2 = -0.012$). The positive coefficient on `Year` enabled the linear model to slightly adjust for temporal drift relative to the static training mean, but neither model explained variance beyond the test set mean.
  * For classification, Chronological accuracy was $34.0\%$ with One-vs-Rest ROC-AUC of $0.497$, while Random accuracy was $35.0\%$ with ROC-AUC of $0.458$.
* **Interpretations and Hypotheses:**
  * Random train/test splits assume independent and identically distributed (i.i.d.) observations, which conceals temporal drift.
  * Chronological splitting provides a far more realistic evaluation of out-of-time forecasting performance for longitudinal and environmental data.

#### 5. Difficulty in Predicting Pollution Severity Classes
* **Direct Observations from the Dataset:**
  * Training partitions were balanced into exact tertiles ($33.3\%$ Low, Medium, High).
  * Test partitions exhibited minor empirical class variations (Random test: 16 Low, 15 Medium, 9 High; Chrono test: 15 Low, 19 Medium, 13 High).
* **Model-Derived Findings:**
  * In the Random split, Logistic Regression failed on the **High severity class** (Precision: $0.00$, Recall: $0.00$, F1: $0.00$), predicting only Low and Medium labels.
  * In the Chronological split, the model achieved poor recall on the **Low class** ($0.13$) and poor precision on the **High class** ($0.17$).
  * Across both splits, classification performance remained weak, with overall accuracy between $34.0\%$ and $35.0\%$ and One-vs-Rest ROC-AUC values near 0.5 ($0.458 - 0.497$).
* **Interpretations and Hypotheses:**
  * The extreme classes (Low and High) were hardest to predict because their distributions overlap substantially across auxiliary predictors (GDP, waste, population, renewable energy).
  * Because `Air_Pollution_Index` was strictly excluded to prevent target leakage, the auxiliary features lacked sufficient discriminative power to establish separable linear decision boundaries.

#### 6. Most Important Limitations of the Dataset
* **Direct Observations from the Dataset:**
  * **Limited Sample Size:** Only $200$ total observations spanning 20 years.
  * **Sparse Country Representation:** 175 unique countries across 200 records; 152 countries appear only once.
  * **Duplicate Key Collision:** Exactly one duplicate compound key `(Spain, 2004)` containing two completely contradictory records across all numerical variables.
  * **Weak Inter-Variable Correlations:** All pairwise correlations among features and targets satisfy $|r| < 0.24$.
* **Model-Derived Findings:**
  * In the random split, regression models underperformed the dummy mean; in the chronological split, they slightly beat the dummy mean on MAE/RMSE but retained negative $R^2$. Logistic classification showed weak performance with AUC near 0.5.
* **Interpretations and Hypotheses:**
  * The dataset exhibits the characteristics of a synthetic or heavily perturbed educational benchmark dataset rather than real-world national accounts, where GDP, industrial waste, and energy consumption typically exhibit strong positive scaling with population.

---

### 10.2 Realistic, Evidence-Based Recommendations

Based on our empirical findings and data auditing, we offer the following recommendations for researchers and data engineers working with environmental datasets:

1. **Transition to Continuous Longitudinal Panels:**
   * Environmental and energy modeling requires consistent panel datasets (repeated observations for the same country cohort over time) rather than sporadic pooled cross-sections. This would allow fixed-effects and dynamic panel models to isolate within-country trajectories from cross-country heterogeneity.
2. **Standardize Measurement Methodologies & Document Provenance:**
   * To prevent conflicting records (such as the Spain 2004 anomaly), environmental data collection bodies should provide clear metadata specifying whether figures represent national aggregates, regional samples, or municipal estimates, along with explicit sensor calibration standards.
3. **Incorporate Direct Waste-to-Energy Operational Indicators:**
   * Future efforts to model waste-to-energy recovery should capture facility-level infrastructure metrics (e.g., number and thermal capacity of municipal waste incinerators, anaerobic digestion plants, tipping fees, feed-in tariffs, and recycling mandates) rather than relying on broad ambient pollution indices.
4. **Collect Localized Spatiotemporal and Meteorological Data for Air Quality:**
   * Predicting air pollution severity requires high-frequency meteorological features (wind speed, atmospheric boundary layer height, temperature inversions) and point-source industrial emission registries, rather than annual macroeconomic aggregates.
5. **Establish Independent Multi-Source Validation:**
   * Before drawing operational conclusions, environmental models should be evaluated on verified independent datasets (such as Eurostat, OECD Environment Statistics, or World Bank Open Data).

---

### 10.3 Limitations and Future Work

#### Core Methodological Limitations:
1. **Sample Size Constraints ($N=200$):** Small sample sizes restrict statistical power and cause high variance in test metrics.
2. **High-Cardinality Entity Space:** With 175 countries across 200 rows, one-hot encoding `Country` would introduce 174 sparse dummy variables, causing severe overfitting and preventing out-of-sample generalization.
3. **The Spain 2004 Anomaly:** Retaining both divergent records preserved raw sample size, but highlighted data collection inconsistencies.
4. **Weak Predictive Performance:** Linear Regression achieved negative out-of-sample $R^2$ across both splits, and Logistic Regression showed weak discriminative performance with ROC-AUC near 0.5.
5. **Ineffectiveness of Linear Feature Engineering:** Derived features failed to improve performance over base features.
6. **Relative Severity Categorization:** Severity labels represent dataset-relative empirical tertiles ($Q_{33\%}$ and $Q_{66\%}$), not statutory or health-based air quality standards.
7. **No Causal Inference:** Predictive associations do not establish causal relationships; correlation does not imply causation.
8. **Exploratory Quadrant Limitations:** The energy recovery opportunity matrix partitions data based on sample medians ($55,299.15$ tons industrial waste and $273.14\text{ GWh}$ energy recovery). This is an exploratory descriptive categorization and **does not establish actual commercial investment opportunities or unused plant capacity**, which require granular facility-level economic and engineering feasibility data.

#### Proposed Future Work:
* **Priority 1: Dataset Expansion:** Focus on collecting larger, verifiable empirical datasets ($N > 2,000$ with continuous country-year coverage) before exploring complex predictive algorithms.
* **Priority 2: Granular Waste Stream Composition:** Disaggregate industrial and plastic waste into specific combustible categories (e.g., municipal solid waste biomass vs. non-recyclable polymers vs. inert construction debris).
* **Priority 3: Domain-Specific Baseline Comparisons:** Benchmark future models against localized physical energy-conversion engineering models rather than standard macroeconomic proxies.